## Text-based Search

In [71]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

In [72]:
import pprint as pp
from opensearchpy import OpenSearch
from opensearchpy import helpers

index_name = OPENSEARCH_USER + '_project'
# Create the client with SSL/TLS enabled, but hostname verification disabled.
client = OpenSearch(
    hosts = [{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

if client.indices.exists(index=index_name):

    resp = client.indices.open(index=index_name)
    print(resp)

    print('\n----------------------------------------------------------------------------------- INDEX SETTINGS')
    settings = client.indices.get_settings(index=index_name)
    pp.pprint(settings)

    print('\n----------------------------------------------------------------------------------- INDEX MAPPINGS')
    mappings = client.indices.get_mapping(index=index_name)
    pp.pprint(mappings)

    print('\n----------------------------------------------------------------------------------- INDEX #DOCs')
    print(client.count(index=index_name))
else:
    print("Index does not exist.")


{'acknowledged': True, 'shards_acknowledged': True}

----------------------------------------------------------------------------------- INDEX SETTINGS
{'uservl07_project': {'settings': {'index': {'creation_date': '1774722072589',
                                             'knn': 'true',
                                             'knn.derived_source': {'enabled': 'true'},
                                             'number_of_replicas': '0',
                                             'number_of_shards': '4',
                                             'provided_name': 'uservl07_project',
                                             'refresh_interval': '1s',
                                             'replication': {'type': 'DOCUMENT'},
                                             'uuid': 'aE6ZFrdMT0mYOXjVBWasuQ',
                                             'version': {'created': '137217827'}}}}}

-------------------------------------------------------------------------------

In [73]:
from datasets import load_dataset
import aiohttp

ds = load_dataset("HuggingFaceM4/COCO", split="validation",
                  storage_options={'client_kwargs': {'timeout': aiohttp.ClientTimeout(total=36000)}}
                  )
print(ds.column_names)
pp.pprint(ds[0])
print(ds[0]['sentences']['raw'])
print(ds[0]['image'].width)


['image', 'filepath', 'sentids', 'filename', 'imgid', 'split', 'sentences', 'cocoid']
{'cocoid': 184613,
 'filename': 'COCO_val2014_000000184613.jpg',
 'filepath': 'COCO_val2014_000000184613.jpg',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x336 at 0x16327FD10>,
 'imgid': 2,
 'sentences': {'imgid': 2,
               'raw': 'A child holding a flowered umbrella and petting a yak.',
               'sentid': 474921,
               'tokens': ['a',
                          'child',
                          'holding',
                          'a',
                          'flowered',
                          'umbrella',
                          'and',
                          'petting',
                          'a',
                          'yak']},
 'sentids': [474921, 479322, 479334, 481560, 483594],
 'split': 'val'}
A child holding a flowered umbrella and petting a yak.
500


Set up an OpenSearch index and define mappings for image identifiers,
captions, and COCO metadata (categories, image dimensions).

In [74]:
#Delete index if necessary

pp.pprint(client.indices.delete(index=index_name))

{'acknowledged': True}


In [75]:
index_body = {
   "settings":{
      "index":{
         "number_of_replicas":0,
         "number_of_shards":4,
         "refresh_interval":"-1",
         "knn":"true"
      },
      "similarity":{
          "bm25-20-75":{
              "type":"BM25",
              "k1":2.0,
              "b":0.75
          },
          "bm25-12-0":{
              "type":"BM25",
              "k1":1.2,
              "b":0.0
          },
          "bm25-0-75":{
              "type":"BM25",
              "k1":0.0,
              "b":0.75
          }

      }
   },
   "mappings":{
       "dynamic":      "strict",
       "properties":{
            "doc_id": { "type": "keyword"},
            "image_id": { "type": "keyword"},
            "caption": { 
                "type": "text",
                "analyzer": "standard",
                "similarity":"BM25",
                "fields":{
                    "bm25_20_75": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-20-75"
                    },
                    "bm25_12_0": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-12-0"
                    },
                    "bm25_0_75": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-0-75"
                     }
                  },
            },
            "categories": { "type": "keyword"},
            "width": { "type": "integer"},
            "height": { "type": "integer"},
      }
   }
}

if client.indices.exists(index=index_name):
    print("Index already existed. Nothing to be done.")
else:        
    response = client.indices.create(index=index_name, body=index_body)
    print('\nCreating index:')
    print(response)



Creating index:
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'uservl07_project'}


Load dataset captions in index

In [76]:
def index_data(client, index_name, dataset):
    actions= []
    for i,item in enumerate(dataset):
        action={
            "doc_id":item['sentences']['sentid'],
            "image_id":item['sentences']['imgid'],
            "caption":item['sentences']['raw'],
            #"categories":
            "width":item['image'].width,
            "height":item['image'].height
        }
        actions.append(action)
        if len(actions) == 1000:
            helpers.bulk(client, actions, index=index_name)
            print(f"Indexed {i+1} documents")
            actions = []
    if actions:
        helpers.bulk(client, actions, index=index_name)
        print(f"Indexed {len(dataset)} documents in total")
count = client.count(index=index_name)
print(count['count'])

if count['count'] == 0:
    index_data(client, index_name, ds)
else:
    print("Index already contains documents. Nothing to be done.")

0
Indexed 1000 documents
Indexed 2000 documents
Indexed 3000 documents
Indexed 4000 documents
Indexed 5000 documents
Indexed 6000 documents
Indexed 7000 documents
Indexed 8000 documents
Indexed 9000 documents
Indexed 10000 documents
Indexed 11000 documents
Indexed 12000 documents
Indexed 13000 documents
Indexed 14000 documents
Indexed 15000 documents
Indexed 16000 documents
Indexed 17000 documents
Indexed 18000 documents
Indexed 19000 documents
Indexed 20000 documents
Indexed 21000 documents
Indexed 22000 documents
Indexed 23000 documents
Indexed 24000 documents
Indexed 25000 documents
Indexed 25010 documents in total


In [77]:
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}}) #set refresh interval to 1s to make all operations available for search after 1s
client.indices.refresh(index=index_name) #refresh immediately

{'_shards': {'total': 4, 'successful': 4, 'failed': 0}}

In [78]:

count = client.count(index=index_name)
print(count['count'])

# check a random document
result = client.search(
    index=index_name,
    body={"query": {"match_all": {}}, "size": 1}
)
print(result['hits']['hits'][0]['_source'])

25010
{'doc_id': 474921, 'image_id': 2, 'caption': 'A child holding a flowered umbrella and petting a yak.', 'width': 500, 'height': 336}


Perform the same query computing similarity with BM25 with different parameters.
For field boosting, idea is to use another field, but the dataset doesn't have a categories objects

In [83]:
query_txt = ds[0]['sentences']['raw']


bm25_fields = ["caption", "caption.bm25_20_75", "caption.bm25_12_0", "caption.bm25_0_75"]

for field in bm25_fields:
    query ={
        "size": 5,
        "query": {
            "match": {
                field: {
                    "query": query_txt
                }
            }
        }
    }
    response = client.search(body=query, index=index_name)
    print(f'\nSearch results for field "{field}":')
    for hit in response['hits']['hits']:
        print(f"Score: {hit['_score']:.4f}, Doc ID: {hit['_source']['doc_id']}, Caption: {hit['_source']['caption']}")


Search results for field "caption":
Score: 17.5575, Doc ID: 474921, Caption: A child holding a flowered umbrella and petting a yak.
Score: 5.7494, Doc ID: 237293, Caption: A woman with child pack on holding an umbrella.
Score: 5.6963, Doc ID: 192463, Caption: a close up of a child holding a closed umbrella
Score: 5.6960, Doc ID: 824983, Caption: Someone is petting a horse while holding a baby.
Score: 5.3039, Doc ID: 402327, Caption: A woman with a black umbrella and a child with a red and black umbrella.

Search results for field "caption.bm25_20_75":
Score: 12.9609, Doc ID: 474921, Caption: A child holding a flowered umbrella and petting a yak.
Score: 4.2744, Doc ID: 237293, Caption: A woman with child pack on holding an umbrella.
Score: 4.2512, Doc ID: 824983, Caption: Someone is petting a horse while holding a baby.
Score: 4.2257, Doc ID: 192463, Caption: a close up of a child holding a closed umbrella
Score: 4.0021, Doc ID: 402327, Caption: A woman with a black umbrella and a chil

In [56]:
client.indices.close(index=index_name)

{'acknowledged': True,
 'shards_acknowledged': True,
 'indices': {'uservl07_project': {'closed': True}}}